# 09-Colab - M5: YOLOv8n-cls Drowsiness Detection (GPU Training)

**Run this notebook on Google Colab with GPU runtime.**

## Before Running
1. Upload your `yolo_frames/` folder (832 MB) to Google Drive at:
   `My Drive/drowsiness_project/yolo_frames/`
2. The folder must contain:
   - `metadata.csv`
   - Subfolders like `A_A/Alert/*.jpg`, `A_D/Drowsy/*.jpg`, etc.
3. Set Colab runtime to **GPU**: Runtime > Change runtime type > T4 GPU

## Pipeline
```
yolo_frames/ (pre-extracted 224x224 IR face frames, 1fps)
  -> Build 5-fold subject-independent CV structure
  -> Train YOLOv8n-cls (ImageNet pretrained, transfer learning)
  -> Evaluate with correct class-index mapping
  -> Save best.pt weights + results JSON
```

| Class | KSS Range | Index |
|-------|-----------|-------|
| Alert | 1-3 | 0 |
| LowVigilant | 4-6 | 1 |
| Drowsy | 7-9 | 2 |

In [ ]:
# Cell 1: SETUP - Install packages, check GPU, mount Drive
!pip install -q ultralytics

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: NO GPU detected! Go to Runtime > Change runtime type > T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: CONFIGURATION - All paths, constants, fold definitions
import os, sys, shutil, json, time, warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from ultralytics import YOLO
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, precision_score, recall_score)

# --- Paths ---
DRIVE_FRAMES = Path('/content/drive/MyDrive/drowsiness_project/yolo_frames')

LOCAL_ROOT   = Path('/content/m5_yolo')
FRAME_DIR    = LOCAL_ROOT / 'yolo_frames'
YOLO_DIR     = LOCAL_ROOT / 'yolo_cls'
CKPT_DIR     = LOCAL_ROOT / 'checkpoints'
RESULT_DIR   = LOCAL_ROOT / 'results'

DRIVE_OUTPUT = Path('/content/drive/MyDrive/drowsiness_project/m5_output')

for d in [LOCAL_ROOT, FRAME_DIR, YOLO_DIR, CKPT_DIR, RESULT_DIR, DRIVE_OUTPUT]:
    d.mkdir(parents=True, exist_ok=True)

# --- Constants (IDENTICAL to notebook 09) ---
SUBJECTS     = list('ABCDEFGHIJKLMNOPQRS')
NO_VIDEO     = {'B', 'I', 'M'}
SESSION_MAP  = {'Alert': 'A', 'Drowsy': 'D'}

CLASS_NAMES  = ['Alert', 'LowVigilant', 'Drowsy']
N_CLASSES    = 3

# --- 5-Fold subject-independent splits (IDENTICAL to notebook 09) ---
FOLDS = [
    list('ABCD'),   # Fold 0
    list('EFGH'),   # Fold 1
    list('IJK'),    # Fold 2
    list('LMNO'),   # Fold 3
    list('PQRS'),   # Fold 4
]

# --- Training hyperparameters ---
YOLO_BASE    = 'yolov8n-cls.pt'
YOLO_EPOCHS  = 20
YOLO_BATCH   = 64
YOLO_IMGSZ   = 224
YOLO_LR0     = 1e-3
YOLO_LRF     = 0.01
YOLO_DROPOUT = 0.3
YOLO_PATIENCE = 15
MODEL_NAME   = 'M5'

print('Configuration ready.')
print(f'  DRIVE_FRAMES : {DRIVE_FRAMES}')
print(f'  LOCAL_ROOT   : {LOCAL_ROOT}')
if torch.cuda.is_available():
    print(f'  DEVICE       : cuda')
else:
    print(f'  DEVICE       : cpu')
print(f'  BATCH SIZE   : {YOLO_BATCH}')
print(f'  FOLDS        : {len(FOLDS)}')

In [ ]:
# Cell 3: COPY FRAMES - Drive to Colab local SSD (much faster I/O)
# This copies ~832 MB. Takes ~2-3 minutes. Skips if already copied.

metadata_src = DRIVE_FRAMES / 'metadata.csv'
metadata_dst = FRAME_DIR / 'metadata.csv'

if not metadata_src.exists():
    raise FileNotFoundError(
        'metadata.csv not found at ' + str(metadata_src) + '\n'
        'Please upload yolo_frames/ folder to Google Drive at:\n'
        '  My Drive/drowsiness_project/yolo_frames/'
    )

existing_jpgs = list(FRAME_DIR.rglob('*.jpg'))
if len(existing_jpgs) > 60000:
    print('Frames already copied (' + str(len(existing_jpgs)) + ' jpg files)')
else:
    print('Copying frames from Drive to local SSD ...')
    t0 = time.time()

    for subdir in sorted(DRIVE_FRAMES.iterdir()):
        if subdir.is_dir():
            dst = FRAME_DIR / subdir.name
            if dst.exists() and len(list(dst.rglob('*.jpg'))) > 100:
                print('  ' + subdir.name + ' -- already copied, skipping')
                continue
            print('  Copying ' + subdir.name + ' ...', end=' ')
            shutil.copytree(str(subdir), str(dst), dirs_exist_ok=True)
            n = len(list(dst.rglob('*.jpg')))
            print(str(n) + ' files')

    shutil.copy2(str(metadata_src), str(metadata_dst))

    elapsed = time.time() - t0
    total = len(list(FRAME_DIR.rglob('*.jpg')))
    print('\nCopied ' + str(total) + ' frames in ' + str(int(elapsed)) + 's')

# Verify
meta_df = pd.read_csv(metadata_dst)
print('\nMetadata: ' + str(len(meta_df)) + ' rows')
print('Subjects: ' + str(sorted(meta_df['subject'].unique())))
print('Classes : ' + str(dict(meta_df['class_name'].value_counts())))

In [ ]:
# Cell 4: FIX METADATA PATHS - Remap Windows paths to Colab paths

meta_df = pd.read_csv(FRAME_DIR / 'metadata.csv')

def remap_path(row):
    original_path = row['path']
    filename = original_path.replace('\\', '/').split('/')[-1]
    new_path = FRAME_DIR / (row['subject'] + '_' + row['session']) / row['class_name'] / filename
    return str(new_path)

meta_df['path'] = meta_df.apply(remap_path, axis=1)

# Verify a few paths actually exist
sample_paths = meta_df['path'].sample(10, random_state=42)
exists_count = sum(Path(p).exists() for p in sample_paths)
print('Path verification: ' + str(exists_count) + '/10 sample paths exist')
if exists_count < 8:
    print('WARNING: Many paths missing! Check that yolo_frames copied correctly.')
    for p in list(sample_paths)[:3]:
        print('  ' + p + '  exists=' + str(Path(p).exists()))
else:
    print('Paths remapped successfully')

meta_df.to_csv(FRAME_DIR / 'metadata.csv', index=False)
print('\nTotal frames : ' + str(len(meta_df)))
print(meta_df[['subject', 'session', 'class', 'class_name']].head())

In [ ]:
# Cell 5: DATASET OVERVIEW - Class distribution + per-fold statistics

meta_df = pd.read_csv(FRAME_DIR / 'metadata.csv')

print('Total frames: ' + str(len(meta_df)))
print('Subjects with frames: ' + str(sorted(meta_df['subject'].unique())))
print('Number of subjects  : ' + str(meta_df['subject'].nunique()))
print()

cls_counts = meta_df['class_name'].value_counts().reindex(CLASS_NAMES)
print('Overall class distribution:')
for cn, cnt in cls_counts.items():
    pct = cnt / len(meta_df) * 100
    print('  ' + cn.ljust(15) + ': ' + str(cnt).rjust(6) + '  (' + f'{pct:.1f}' + '%)')

print('\n' + '---' * 20)
print('Per-fold split sizes:')
for fold_idx, test_subjects in enumerate(FOLDS):
    test_subs  = [s for s in test_subjects if s not in NO_VIDEO]
    train_subs = [s for s in SUBJECTS if s not in NO_VIDEO and s not in test_subjects]
    test_df  = meta_df[meta_df['subject'].isin(test_subs)]
    train_df = meta_df[meta_df['subject'].isin(train_subs)]
    print('  Fold ' + str(fold_idx) + ': train=' + str(len(train_df)) +
          ' (' + str(len(train_subs)) + ' subj)  test=' + str(len(test_df)) +
          ' (' + str(len(test_subs)) + ' subj)  test_subj=' + str(test_subs))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']

axes[0].bar(CLASS_NAMES, cls_counts.values, color=colors, edgecolor='white')
for i, cnt in enumerate(cls_counts.values):
    axes[0].text(i, cnt + 50, str(cnt), ha='center', fontsize=10)
axes[0].set_title('Overall Class Distribution')
axes[0].set_ylabel('Frame count')

pivot = meta_df.groupby(['subject', 'class_name']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=CLASS_NAMES, fill_value=0)
pivot.plot.bar(stacked=True, ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Frames per Subject')
axes[1].set_ylabel('Frame count')
axes[1].legend(title='Class')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: BUILD FOLD DIRECTORIES - Symlink frames into YOLO structure
# Creates:  yolo_cls/fold_X/train/{Alert,LowVigilant,Drowsy}/*.jpg
#           yolo_cls/fold_X/val/{Alert,LowVigilant,Drowsy}/*.jpg

meta_df = pd.read_csv(FRAME_DIR / 'metadata.csv')

total_t0 = time.time()

for fold_idx, test_subjects in enumerate(FOLDS):
    fold_dir = YOLO_DIR / ('fold_' + str(fold_idx))

    existing = sum(1 for _ in (fold_dir / 'train').rglob('*.jpg')) if (fold_dir / 'train').exists() else 0
    if existing > 100:
        val_count = sum(1 for _ in (fold_dir / 'val').rglob('*.jpg')) if (fold_dir / 'val').exists() else 0
        print('Fold ' + str(fold_idx) + ': already built (' + str(existing) + ' train + ' + str(val_count) + ' val) -- skipping')
        continue

    test_subs  = [s for s in test_subjects if s not in NO_VIDEO]
    train_subs = [s for s in SUBJECTS if s not in NO_VIDEO and s not in test_subjects]

    test_df  = meta_df[meta_df['subject'].isin(test_subs)]
    train_df = meta_df[meta_df['subject'].isin(train_subs)]

    for split in ['train', 'val']:
        for cn in CLASS_NAMES:
            (fold_dir / split / cn).mkdir(parents=True, exist_ok=True)

    def link_frames(df, split):
        linked = 0
        for _, row in df.iterrows():
            src = Path(row['path'])
            if not src.exists():
                continue
            dst_name = row['subject'] + '_' + row['session'] + '_' + src.name
            dst = fold_dir / split / row['class_name'] / dst_name
            if not dst.exists():
                try:
                    os.symlink(str(src), str(dst))
                except OSError:
                    shutil.copy2(str(src), str(dst))
            linked += 1
        return linked

    n_tr = link_frames(train_df, 'train')
    n_te = link_frames(test_df,  'val')
    print('Fold ' + str(fold_idx) + ': linked ' + str(n_tr) + ' train + ' + str(n_te) + ' val  |  test_subj=' + str(test_subs))

elapsed = time.time() - total_t0
print('\nAll folds built in ' + str(int(elapsed)) + 's')

print('\nVerification:')
for fold_idx in range(len(FOLDS)):
    fold_dir = YOLO_DIR / ('fold_' + str(fold_idx))
    for split in ['train', 'val']:
        counts = {}
        for cn in CLASS_NAMES:
            d = fold_dir / split / cn
            counts[cn] = len(list(d.glob('*.jpg'))) if d.exists() else 0
        total = sum(counts.values())
        print('  Fold ' + str(fold_idx) + '/' + split + ': ' + str(total) + ' total  ' + str(counts))

In [ ]:
# Cell 7: TRAIN ALL 5 FOLDS - YOLOv8n-cls with GPU
# Expected time on T4 GPU: ~5-8 min per fold, ~30-40 min total
#
# CRITICAL: YOLO class-index mapping
#   YOLO sorts class folders ALPHABETICALLY, so:
#     YOLO index 0 -> Alert        (matches our index 0)
#     YOLO index 1 -> Drowsy       (our index is 2!)
#     YOLO index 2 -> LowVigilant  (our index is 1!)
#   We handle this in evaluation (Cell 8) using model.names dict.

training_log = []

for fold_idx in range(len(FOLDS)):
    fold_dir = YOLO_DIR / ('fold_' + str(fold_idx))
    best_pt  = CKPT_DIR / ('M5_fold' + str(fold_idx)) / 'weights' / 'best.pt'

    print('\n' + '=' * 60)
    print('  FOLD ' + str(fold_idx) + '  |  test subjects = ' + str(FOLDS[fold_idx]))
    print('=' * 60)

    if best_pt.exists():
        print('  best.pt already exists -- SKIPPING training')
        print('    ' + str(best_pt))
        training_log.append({'fold': fold_idx, 'status': 'skipped', 'time_min': 0})
        continue

    t0 = time.time()
    print('  Training YOLOv8n-cls ...')
    print('    epochs=' + str(YOLO_EPOCHS) + '  batch=' + str(YOLO_BATCH) + '  imgsz=' + str(YOLO_IMGSZ))
    print('    lr0=' + str(YOLO_LR0) + '  lrf=' + str(YOLO_LRF) + '  dropout=' + str(YOLO_DROPOUT))

    model = YOLO(YOLO_BASE)
    results = model.train(
        data      = str(fold_dir),
        epochs    = YOLO_EPOCHS,
        imgsz     = YOLO_IMGSZ,
        batch     = YOLO_BATCH,
        project   = str(CKPT_DIR),
        name      = 'M5_fold' + str(fold_idx),
        exist_ok  = True,
        verbose   = True,
        patience  = YOLO_PATIENCE,
        lr0       = YOLO_LR0,
        lrf       = YOLO_LRF,
        dropout   = YOLO_DROPOUT,
        device    = 0,
        workers   = 2,
    )

    elapsed_min = (time.time() - t0) / 60
    print('  Fold ' + str(fold_idx) + ' training complete in ' + f'{elapsed_min:.1f}' + ' min')

    if best_pt.exists():
        size_mb = best_pt.stat().st_size / 1e6
        print('  best.pt saved (' + f'{size_mb:.1f}' + ' MB): ' + str(best_pt))
    else:
        print('  WARNING: best.pt NOT found at ' + str(best_pt))
        for p in CKPT_DIR.rglob('best.pt'):
            print('    Found: ' + str(p))

    training_log.append({'fold': fold_idx, 'status': 'trained', 'time_min': elapsed_min})

print('\n' + '=' * 60)
print('Training Summary:')
total_time = 0
for entry in training_log:
    total_time += entry['time_min']
    print('  Fold ' + str(entry['fold']) + ': ' + entry['status'].ljust(8) + '  (' + f'{entry["time_min"]:.1f}' + ' min)')
print('  Total training time: ' + f'{total_time:.1f}' + ' min')
print('=' * 60)

In [ ]:
# Cell 8: EVALUATE ALL FOLDS - With correct class-index mapping
#
# KEY INSIGHT: YOLO sorts class directories ALPHABETICALLY:
#     YOLO index 0 -> 'Alert'
#     YOLO index 1 -> 'Drowsy'       <- NOT LowVigilant!
#     YOLO index 2 -> 'LowVigilant'  <- NOT Drowsy!
#
# Our CLASS_NAMES = ['Alert', 'LowVigilant', 'Drowsy'] uses:
#     Our index 0 -> Alert
#     Our index 1 -> LowVigilant
#     Our index 2 -> Drowsy
#
# So we use model.names to build a mapping from YOLO indices to our indices.

meta_df = pd.read_csv(FRAME_DIR / 'metadata.csv')
all_fold_results = []

for fold_idx in range(len(FOLDS)):
    best_pt = CKPT_DIR / ('M5_fold' + str(fold_idx)) / 'weights' / 'best.pt'

    if not best_pt.exists():
        print('Fold ' + str(fold_idx) + ': best.pt not found -- skipping evaluation')
        continue

    model = YOLO(str(best_pt))

    # Build class-index mapping: YOLO_idx -> our_idx
    # model.names = {0: 'Alert', 1: 'Drowsy', 2: 'LowVigilant'}
    yolo_to_ours = {}
    for yolo_idx, yolo_name in model.names.items():
        our_idx = CLASS_NAMES.index(yolo_name)
        yolo_to_ours[yolo_idx] = our_idx

    print('\n' + '---' * 20)
    print('Fold ' + str(fold_idx) + '  |  test subjects = ' + str(FOLDS[fold_idx]))
    print('  YOLO class mapping: ' + str(model.names))
    print('  YOLO->Ours mapping: ' + str(yolo_to_ours))
    print('  Our CLASS_NAMES   : ' + str(CLASS_NAMES))

    fold_dir = YOLO_DIR / ('fold_' + str(fold_idx))
    y_true, y_pred = [], []

    for cn_idx, cn in enumerate(CLASS_NAMES):
        class_dir = fold_dir / 'val' / cn
        if not class_dir.exists():
            continue
        img_list = sorted(class_dir.glob('*.jpg'))
        if not img_list:
            continue

        preds = model.predict(
            source=str(class_dir),
            verbose=False,
            batch=YOLO_BATCH,
        )
        for p in preds:
            yolo_pred_idx = int(p.probs.top1)
            our_pred_idx  = yolo_to_ours[yolo_pred_idx]
            y_true.append(cn_idx)
            y_pred.append(our_pred_idx)

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)

    print('  Accuracy  : ' + f'{acc:.4f}')
    print('  Macro-F1  : ' + f'{f1:.4f}')
    print('  Precision : ' + f'{prec:.4f}')
    print('  Recall    : ' + f'{rec:.4f}')
    print('  Samples   : ' + str(len(y_true)))

    print(classification_report(
        y_true, y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    print('  Confusion matrix:')
    print('               Pred_Alert  Pred_LowVig  Pred_Drowsy')
    for i, cn_label in enumerate(CLASS_NAMES):
        print('  True_' + cn_label.ljust(11) + '  ' + str(cm[i,0]).rjust(7) + '  ' + str(cm[i,1]).rjust(11) + '  ' + str(cm[i,2]).rjust(11))

    test_subs = [s for s in FOLDS[fold_idx] if s not in NO_VIDEO]
    fold_result = {
        'fold': fold_idx,
        'accuracy': acc,
        'macro_f1': f1,
        'precision': prec,
        'recall': rec,
        'y_true': y_true,
        'y_pred': y_pred,
        'test_subjects': test_subs,
        'n_train': len(meta_df[~meta_df['subject'].isin(FOLDS[fold_idx])]),
        'n_test': len(y_true),
    }
    all_fold_results.append(fold_result)

# Aggregate results
if all_fold_results:
    accs = [f['accuracy'] for f in all_fold_results]
    f1s  = [f['macro_f1'] for f in all_fold_results]
    precs = [f['precision'] for f in all_fold_results]
    recs  = [f['recall'] for f in all_fold_results]

    results = {
        'model_name': MODEL_NAME,
        'mean_acc': float(np.mean(accs)),
        'std_acc':  float(np.std(accs)),
        'mean_f1':  float(np.mean(f1s)),
        'std_f1':   float(np.std(f1s)),
        'mean_precision': float(np.mean(precs)),
        'mean_recall': float(np.mean(recs)),
        'folds': all_fold_results,
    }

    print('\n' + '=' * 60)
    print('=== M5 YOLOv8-cls Aggregate Results ===')
    print('Accuracy  : ' + f'{results["mean_acc"]:.4f}' + ' +/- ' + f'{results["std_acc"]:.4f}')
    print('Macro-F1  : ' + f'{results["mean_f1"]:.4f}' + ' +/- ' + f'{results["std_f1"]:.4f}')
    print('Precision : ' + f'{results["mean_precision"]:.4f}')
    print('Recall    : ' + f'{results["mean_recall"]:.4f}')
    print('=' * 60)
else:
    print('\nNo folds evaluated! Check that best.pt files exist.')

In [ ]:
# Cell 9: VISUALIZATIONS - Bar charts + Confusion matrices

if not all_fold_results:
    print('No results to visualize.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    folds_x = ['Fold ' + str(f['fold']) for f in all_fold_results]
    accs    = [f['accuracy'] for f in all_fold_results]
    f1s     = [f['macro_f1'] for f in all_fold_results]

    axes[0].bar(folds_x, accs, color='steelblue', edgecolor='white')
    axes[0].axhline(np.mean(accs), color='red', ls='--', label='Mean=' + f'{np.mean(accs):.3f}')
    for i, v in enumerate(accs):
        axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    axes[0].set_title('M5 -- Per-fold Accuracy')
    axes[0].set_ylim(0, 1)
    axes[0].legend()

    axes[1].bar(folds_x, f1s, color='darkorange', edgecolor='white')
    axes[1].axhline(np.mean(f1s), color='red', ls='--', label='Mean=' + f'{np.mean(f1s):.3f}')
    for i, v in enumerate(f1s):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    axes[1].set_title('M5 -- Per-fold Macro-F1')
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(RESULT_DIR / 'M5_fold_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Aggregate confusion matrix
    all_true = []
    all_pred = []
    for f in all_fold_results:
        all_true.extend(f['y_true'])
        all_pred.extend(f['y_pred'])

    cm = confusion_matrix(all_true, all_pred, labels=[0, 1, 2])

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    axes[0].set_title('M5 -- Confusion Matrix (all folds)')

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    axes[1].set_title('M5 -- Normalised Confusion Matrix')

    plt.tight_layout()
    plt.savefig(RESULT_DIR / 'M5_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 10: RESULTS TABLE - Per-fold + aggregate summary

if not all_fold_results:
    print('No results available.')
else:
    rows = []
    for f in all_fold_results:
        rows.append({
            'Fold': f['fold'],
            'Test Subjects': ', '.join(f['test_subjects']),
            'Train': f['n_train'],
            'Test': f['n_test'],
            'Accuracy': f'{f["accuracy"]:.4f}',
            'Macro-F1': f'{f["macro_f1"]:.4f}',
            'Precision': f'{f["precision"]:.4f}',
            'Recall': f'{f["recall"]:.4f}',
        })

    df_results = pd.DataFrame(rows)

    accs = [f['accuracy'] for f in all_fold_results]
    f1s  = [f['macro_f1'] for f in all_fold_results]
    precs = [f['precision'] for f in all_fold_results]
    recs  = [f['recall'] for f in all_fold_results]

    mean_row = pd.DataFrame([{
        'Fold': 'Mean', 'Test Subjects': '',
        'Train': '', 'Test': '',
        'Accuracy': f'{np.mean(accs):.4f}',
        'Macro-F1': f'{np.mean(f1s):.4f}',
        'Precision': f'{np.mean(precs):.4f}',
        'Recall': f'{np.mean(recs):.4f}',
    }])
    std_row = pd.DataFrame([{
        'Fold': 'Std', 'Test Subjects': '',
        'Train': '', 'Test': '',
        'Accuracy': f'{np.std(accs):.4f}',
        'Macro-F1': f'{np.std(f1s):.4f}',
        'Precision': f'{np.std(precs):.4f}',
        'Recall': f'{np.std(recs):.4f}',
    }])

    df_results = pd.concat([df_results, mean_row, std_row], ignore_index=True)
    display(df_results)

In [ ]:
# Cell 11: SAVE RESULTS + WEIGHTS to Google Drive

def to_serializable(obj):
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, dict):           return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):           return [to_serializable(i) for i in obj]
    return obj

if not all_fold_results:
    print('No results to save.')
else:
    # Save JSON locally
    json_path = RESULT_DIR / (MODEL_NAME + '_results.json')
    with open(json_path, 'w') as f:
        json.dump(to_serializable(results), f, indent=2)
    print('Results JSON saved: ' + str(json_path))

    # Save to Drive
    drive_json = DRIVE_OUTPUT / (MODEL_NAME + '_results.json')
    shutil.copy2(str(json_path), str(drive_json))
    print('Results JSON saved to Drive: ' + str(drive_json))

    # Copy best.pt weights to Drive
    weights_dir = DRIVE_OUTPUT / 'weights'
    weights_dir.mkdir(parents=True, exist_ok=True)

    for fold_idx in range(len(FOLDS)):
        src = CKPT_DIR / ('M5_fold' + str(fold_idx)) / 'weights' / 'best.pt'
        if src.exists():
            dst = weights_dir / ('M5_fold' + str(fold_idx) + '_best.pt')
            shutil.copy2(str(src), str(dst))
            size_mb = dst.stat().st_size / 1e6
            print('Fold ' + str(fold_idx) + ' best.pt (' + f'{size_mb:.1f}' + ' MB) saved to Drive')
        else:
            print('Fold ' + str(fold_idx) + ' best.pt not found')

    # Copy plots to Drive
    for png in RESULT_DIR.glob('*.png'):
        shutil.copy2(str(png), str(DRIVE_OUTPUT / png.name))
        print(png.name + ' saved to Drive')

    print('\n' + '=' * 60)
    print('All outputs saved to: ' + str(DRIVE_OUTPUT))
    print('\nTo use on your local PC:')
    print('  1. Download weights from Drive: ' + str(weights_dir))
    print('  2. Place M5_foldX_best.pt in local:')
    print('     models/checkpoints/yolo_cls/M5_foldX/weights/best.pt')
    print('  3. Download M5_results.json to results/reports/')
    print('=' * 60)

In [ ]:
# Cell 12: VERIFY - Quick sanity check on saved weights

print('Verifying saved weights on Drive ...')
weights_dir = DRIVE_OUTPUT / 'weights'

for fold_idx in range(len(FOLDS)):
    pt_path = weights_dir / ('M5_fold' + str(fold_idx) + '_best.pt')
    if pt_path.exists():
        model = YOLO(str(pt_path))
        print('  Fold ' + str(fold_idx) + ': loaded  |  classes=' + str(model.names))
    else:
        print('  Fold ' + str(fold_idx) + ': missing')

print('\n=== Final M5 Results ===')
print('Accuracy  : ' + f'{results["mean_acc"]:.4f}' + ' +/- ' + f'{results["std_acc"]:.4f}')
print('Macro-F1  : ' + f'{results["mean_f1"]:.4f}' + ' +/- ' + f'{results["std_f1"]:.4f}')
print('Precision : ' + f'{results["mean_precision"]:.4f}')
print('Recall    : ' + f'{results["mean_recall"]:.4f}')
print('\nDone! Download weights + results from Google Drive.')

# 09-Colab — M5: YOLOv8n-cls Drowsiness Detection (GPU Training)

**Run this notebook on Google Colab with GPU runtime.**

## Before Running
1. Upload your `yolo_frames/` folder (832 MB) to Google Drive at:  
   `My Drive/drowsiness_project/yolo_frames/`
2. The folder must contain:
   - `metadata.csv` — frame metadata with columns: subject, session, class, class_name, path
   - Subfolders like `A_A/Alert/*.jpg`, `A_D/Drowsy/*.jpg`, etc.
3. Set Colab runtime to **GPU**: Runtime → Change runtime type → T4 GPU

## Pipeline
```
yolo_frames/ (pre-extracted 224×224 IR face frames, 1fps)
  → Build 5-fold subject-independent CV structure
  → Train YOLOv8n-cls (ImageNet pretrained, transfer learning)
  → Evaluate with correct class-index mapping
  → Save best.pt weights + results JSON
```

| Class | KSS Range | Index |
|-------|-----------|-------|
| Alert | ≤ 3 | 0 |
| LowVigilant | 4–6 | 1 |
| Drowsy | > 6 | 2 |

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: SETUP — Install packages, check GPU, mount Drive
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q ultralytics

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠ NO GPU detected! Go to Runtime → Change runtime type → T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

PyTorch : 2.10.0+cpu
CUDA    : False
⚠ NO GPU detected! Go to Runtime → Change runtime type → T4 GPU


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2: CONFIGURATION — All paths, constants, fold definitions
# ══════════════════════════════════════════════════════════════════════════════
import os, sys, shutil, json, time, warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from ultralytics import YOLO
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, precision_score, recall_score)

# ── Paths ─────────────────────────────────────────────────────────────────────
# Source: your uploaded frames on Google Drive
DRIVE_FRAMES = Path('/content/drive/MyDrive/drowsiness_project/yolo_frames')

# Local working directories (fast Colab SSD)
LOCAL_ROOT   = Path('/content/m5_yolo')
FRAME_DIR    = LOCAL_ROOT / 'yolo_frames'       # copied from Drive
YOLO_DIR     = LOCAL_ROOT / 'yolo_cls'          # per-fold train/val dirs
CKPT_DIR     = LOCAL_ROOT / 'checkpoints'       # training outputs
RESULT_DIR   = LOCAL_ROOT / 'results'           # JSON + plots

# Output on Drive (persistent across sessions)
DRIVE_OUTPUT = Path('/content/drive/MyDrive/drowsiness_project/m5_output')

for d in [LOCAL_ROOT, FRAME_DIR, YOLO_DIR, CKPT_DIR, RESULT_DIR, DRIVE_OUTPUT]:
    d.mkdir(parents=True, exist_ok=True)

# ── Constants (IDENTICAL to notebook 09) ──────────────────────────────────────
SUBJECTS     = list('ABCDEFGHIJKLMNOPQRS')
NO_VIDEO     = {'B', 'I', 'M'}           # no video files for these subjects
SESSION_MAP  = {'Alert': 'A', 'Drowsy': 'D'}

CLASS_NAMES  = ['Alert', 'LowVigilant', 'Drowsy']   # Our ordering (idx 0,1,2)
N_CLASSES    = 3

# ── 5-Fold subject-independent splits (IDENTICAL to notebook 09) ──────────────
FOLDS = [
    list('ABCD'),   # Fold 0
    list('EFGH'),   # Fold 1
    list('IJK'),    # Fold 2
    list('LMNO'),   # Fold 3
    list('PQRS'),   # Fold 4
]

# ── Training hyperparameters ──────────────────────────────────────────────────
YOLO_BASE    = 'yolov8n-cls.pt'   # ImageNet pretrained
YOLO_EPOCHS  = 20
YOLO_BATCH   = 64                 # larger batch — GPU has enough VRAM
YOLO_IMGSZ   = 224
YOLO_LR0     = 1e-3
YOLO_LRF     = 0.01
YOLO_DROPOUT = 0.3
YOLO_PATIENCE = 15
MODEL_NAME   = 'M5'

print('Configuration ready.')
print(f'  DRIVE_FRAMES : {DRIVE_FRAMES}')
print(f'  LOCAL_ROOT   : {LOCAL_ROOT}')
print(f'  DEVICE       : {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'  BATCH SIZE   : {YOLO_BATCH}')
print(f'  FOLDS        : {len(FOLDS)}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: COPY FRAMES — Drive → Colab local SSD (much faster I/O)
# ══════════════════════════════════════════════════════════════════════════════
# This copies ~832 MB from Drive to local SSD. Takes ~2-3 minutes.
# Skips if already copied (e.g., if you re-run this cell).

metadata_src = DRIVE_FRAMES / 'metadata.csv'
metadata_dst = FRAME_DIR / 'metadata.csv'

if not metadata_src.exists():
    raise FileNotFoundError(
        f'metadata.csv not found at {metadata_src}\n'
        f'Please upload yolo_frames/ folder to Google Drive at:\n'
        f'  My Drive/drowsiness_project/yolo_frames/'
    )

# Check if already copied
existing_jpgs = list(FRAME_DIR.rglob('*.jpg'))
if len(existing_jpgs) > 60000:
    print(f'✓ Frames already copied ({len(existing_jpgs):,} jpg files)')
else:
    print(f'Copying frames from Drive to local SSD ...')
    t0 = time.time()

    # Copy all subject_session folders
    for subdir in sorted(DRIVE_FRAMES.iterdir()):
        if subdir.is_dir():
            dst = FRAME_DIR / subdir.name
            if dst.exists() and len(list(dst.rglob('*.jpg'))) > 100:
                print(f'  {subdir.name} — already copied, skipping')
                continue
            print(f'  Copying {subdir.name} ...', end=' ')
            shutil.copytree(str(subdir), str(dst), dirs_exist_ok=True)
            n = len(list(dst.rglob('*.jpg')))
            print(f'{n} files')

    # Copy metadata.csv
    shutil.copy2(str(metadata_src), str(metadata_dst))

    elapsed = time.time() - t0
    total = len(list(FRAME_DIR.rglob('*.jpg')))
    print(f'\n✓ Copied {total:,} frames in {elapsed:.0f}s')

# Verify
meta_df = pd.read_csv(metadata_dst)
print(f'\nMetadata: {len(meta_df):,} rows')
print(f'Subjects: {sorted(meta_df["subject"].unique())}')
print(f'Classes : {dict(meta_df["class_name"].value_counts())}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: FIX METADATA PATHS — Remap Windows paths → Colab paths
# ══════════════════════════════════════════════════════════════════════════════
# The metadata.csv has Windows absolute paths. We rebuild them from the
# subject, session, class_name columns to point to local Colab copies.

meta_df = pd.read_csv(FRAME_DIR / 'metadata.csv')

def remap_path(row):
    """Rebuild path from subject/session/class_name + original filename."""
    # Extract just the filename from the original Windows path
    original_path = row['path']
    filename = original_path.replace('\\', '/').split('/')[-1]  # e.g. frame_00000.jpg
    # Rebuild: FRAME_DIR / {subject}_{session} / {class_name} / {filename}
    new_path = FRAME_DIR / f"{row['subject']}_{row['session']}" / row['class_name'] / filename
    return str(new_path)

meta_df['path'] = meta_df.apply(remap_path, axis=1)

# Verify a few paths actually exist
sample_paths = meta_df['path'].sample(10, random_state=42)
exists_count = sum(Path(p).exists() for p in sample_paths)
print(f'Path verification: {exists_count}/10 sample paths exist')
if exists_count < 8:
    print('⚠ Many paths missing! Check that yolo_frames copied correctly.')
    print('Sample paths:')
    for p in sample_paths[:3]:
        print(f'  {p}  → exists={Path(p).exists()}')
else:
    print('✓ Paths remapped successfully')

# Save updated metadata
meta_df.to_csv(FRAME_DIR / 'metadata.csv', index=False)
print(f'\nTotal frames : {len(meta_df):,}')
print(meta_df[['subject', 'session', 'class', 'class_name']].head())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5: DATASET OVERVIEW — Class distribution + per-fold statistics
# ══════════════════════════════════════════════════════════════════════════════

meta_df = pd.read_csv(FRAME_DIR / 'metadata.csv')

print(f'Total frames: {len(meta_df):,}')
print(f'Subjects with frames: {sorted(meta_df["subject"].unique())}')
print(f'Number of subjects  : {meta_df["subject"].nunique()}')
print()

# Overall class distribution
cls_counts = meta_df['class_name'].value_counts().reindex(CLASS_NAMES)
print('Overall class distribution:')
for cn, cnt in cls_counts.items():
    print(f'  {cn:15s}: {cnt:6d}  ({cnt/len(meta_df)*100:.1f}%)')

# Per-fold train/test split sizes
print(f'\n{"─"*60}')
print('Per-fold split sizes:')
for fold_idx, test_subjects in enumerate(FOLDS):
    test_subs  = [s for s in test_subjects if s not in NO_VIDEO]
    train_subs = [s for s in SUBJECTS if s not in NO_VIDEO and s not in test_subjects]
    test_df  = meta_df[meta_df['subject'].isin(test_subs)]
    train_df = meta_df[meta_df['subject'].isin(train_subs)]
    print(f'  Fold {fold_idx}: train={len(train_df):,} ({len(train_subs)} subj)  '
          f'test={len(test_df):,} ({len(test_subs)} subj)  '
          f'test_subj={test_subs}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']

axes[0].bar(CLASS_NAMES, cls_counts.values, color=colors, edgecolor='white')
for i, cnt in enumerate(cls_counts.values):
    axes[0].text(i, cnt + 50, str(cnt), ha='center', fontsize=10)
axes[0].set_title('Overall Class Distribution')
axes[0].set_ylabel('Frame count')

pivot = meta_df.groupby(['subject', 'class_name']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=CLASS_NAMES, fill_value=0)
pivot.plot.bar(stacked=True, ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Frames per Subject')
axes[1].set_ylabel('Frame count')
axes[1].legend(title='Class')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6: BUILD FOLD DIRECTORIES — Symlink frames into YOLO structure
# ══════════════════════════════════════════════════════════════════════════════
# Creates:  yolo_cls/fold_X/train/{Alert,LowVigilant,Drowsy}/*.jpg
#           yolo_cls/fold_X/val/{Alert,LowVigilant,Drowsy}/*.jpg
# Uses symlinks (instant, no extra disk space on Linux/Colab).

meta_df = pd.read_csv(FRAME_DIR / 'metadata.csv')

total_t0 = time.time()

for fold_idx, test_subjects in enumerate(FOLDS):
    fold_dir = YOLO_DIR / f'fold_{fold_idx}'

    # Check if already built
    existing = sum(1 for _ in (fold_dir / 'train').rglob('*.jpg')) if (fold_dir / 'train').exists() else 0
    if existing > 100:
        val_count = sum(1 for _ in (fold_dir / 'val').rglob('*.jpg')) if (fold_dir / 'val').exists() else 0
        print(f'Fold {fold_idx}: already built ({existing} train + {val_count} val) — skipping')
        continue

    test_subs  = [s for s in test_subjects if s not in NO_VIDEO]
    train_subs = [s for s in SUBJECTS if s not in NO_VIDEO and s not in test_subjects]

    test_df  = meta_df[meta_df['subject'].isin(test_subs)]
    train_df = meta_df[meta_df['subject'].isin(train_subs)]

    # Create directory structure
    for split in ['train', 'val']:
        for cn in CLASS_NAMES:
            (fold_dir / split / cn).mkdir(parents=True, exist_ok=True)

    # Symlink frames
    def link_frames(df, split):
        linked = 0
        for _, row in df.iterrows():
            src = Path(row['path'])
            if not src.exists():
                continue
            # Unique filename: {subject}_{session}_{original_name}
            dst_name = f"{row['subject']}_{row['session']}_{src.name}"
            dst = fold_dir / split / row['class_name'] / dst_name
            if not dst.exists():
                try:
                    os.symlink(str(src), str(dst))  # symlink on Linux (Colab)
                except OSError:
                    shutil.copy2(str(src), str(dst))  # fallback to copy
            linked += 1
        return linked

    n_tr = link_frames(train_df, 'train')
    n_te = link_frames(test_df,  'val')
    print(f'Fold {fold_idx}: linked {n_tr} train + {n_te} val  |  test_subj={test_subs}')

elapsed = time.time() - total_t0
print(f'\n✓ All folds built in {elapsed:.0f}s')

# Verify class counts per fold
print(f'\nVerification:')
for fold_idx in range(len(FOLDS)):
    fold_dir = YOLO_DIR / f'fold_{fold_idx}'
    for split in ['train', 'val']:
        counts = {}
        for cn in CLASS_NAMES:
            d = fold_dir / split / cn
            counts[cn] = len(list(d.glob('*.jpg'))) if d.exists() else 0
        total = sum(counts.values())
        print(f'  Fold {fold_idx}/{split}: {total:,} total  {counts}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: TRAIN ALL 5 FOLDS — YOLOv8n-cls with GPU
# ══════════════════════════════════════════════════════════════════════════════
# Expected time on T4 GPU: ~5-8 min per fold → ~30-40 min total
#
# ⚠ CRITICAL: YOLO class-index mapping
#   YOLO sorts class folders ALPHABETICALLY, so:
#     YOLO index 0 → Alert        (matches our index 0) ✓
#     YOLO index 1 → Drowsy       (our index is 2!)     ✗
#     YOLO index 2 → LowVigilant  (our index is 1!)     ✗
#   We handle this in evaluation (Cell 8) using model.names dict.

training_log = []

for fold_idx in range(len(FOLDS)):
    fold_dir = YOLO_DIR / f'fold_{fold_idx}'
    best_pt  = CKPT_DIR / f'M5_fold{fold_idx}' / 'weights' / 'best.pt'

    print(f'\n{"="*60}')
    print(f'  FOLD {fold_idx}  |  test subjects = {FOLDS[fold_idx]}')
    print(f'{"="*60}')

    if best_pt.exists():
        print(f'  ✓ best.pt already exists — SKIPPING training')
        print(f'    {best_pt}')
        training_log.append({'fold': fold_idx, 'status': 'skipped', 'time_min': 0})
        continue

    # ── Train ─────────────────────────────────────────────────────────────
    t0 = time.time()
    print(f'  Training YOLOv8n-cls ...')
    print(f'    epochs={YOLO_EPOCHS}  batch={YOLO_BATCH}  imgsz={YOLO_IMGSZ}')
    print(f'    lr0={YOLO_LR0}  lrf={YOLO_LRF}  dropout={YOLO_DROPOUT}')

    model = YOLO(YOLO_BASE)
    results = model.train(
        data      = str(fold_dir),
        epochs    = YOLO_EPOCHS,
        imgsz     = YOLO_IMGSZ,
        batch     = YOLO_BATCH,
        project   = str(CKPT_DIR),
        name      = f'M5_fold{fold_idx}',
        exist_ok  = True,
        verbose   = True,
        patience  = YOLO_PATIENCE,
        lr0       = YOLO_LR0,
        lrf       = YOLO_LRF,
        dropout   = YOLO_DROPOUT,
        device    = 0,            # GPU 0
        workers   = 2,            # Colab has limited CPU workers
    )

    elapsed_min = (time.time() - t0) / 60
    print(f'  ✓ Fold {fold_idx} training complete in {elapsed_min:.1f} min')

    # Verify best.pt was created
    if best_pt.exists():
        size_mb = best_pt.stat().st_size / 1e6
        print(f'  ✓ best.pt saved ({size_mb:.1f} MB): {best_pt}')
    else:
        print(f'  ✗ WARNING: best.pt NOT found at {best_pt}')
        # Try to find it
        for p in CKPT_DIR.rglob('best.pt'):
            print(f'    Found: {p}')

    training_log.append({'fold': fold_idx, 'status': 'trained', 'time_min': elapsed_min})

# ── Summary ──────────────────────────────────────────────────────────────────
print(f'\n{"═"*60}')
print('Training Summary:')
total_time = 0
for entry in training_log:
    status = entry['status']
    t = entry['time_min']
    total_time += t
    print(f'  Fold {entry["fold"]}: {status:8s}  ({t:.1f} min)')
print(f'  Total training time: {total_time:.1f} min')
print(f'{"═"*60}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: EVALUATE ALL FOLDS — With correct class-index mapping
# ══════════════════════════════════════════════════════════════════════════════
#
# ⚠ KEY INSIGHT: YOLO sorts class directories ALPHABETICALLY:
#     YOLO index 0 → 'Alert'
#     YOLO index 1 → 'Drowsy'       ← NOT LowVigilant!
#     YOLO index 2 → 'LowVigilant'  ← NOT Drowsy!
#
# Our CLASS_NAMES = ['Alert', 'LowVigilant', 'Drowsy'] uses:
#     Our index 0 → Alert
#     Our index 1 → LowVigilant
#     Our index 2 → Drowsy
#
# So we use model.names to build a mapping from YOLO indices → our indices.

all_fold_results = []

for fold_idx in range(len(FOLDS)):
    best_pt = CKPT_DIR / f'M5_fold{fold_idx}' / 'weights' / 'best.pt'

    if not best_pt.exists():
        print(f'Fold {fold_idx}: best.pt not found — skipping evaluation')
        continue

    # Load the trained model
    model = YOLO(str(best_pt))

    # ── Build class-index mapping ─────────────────────────────────────────
    # model.names = {0: 'Alert', 1: 'Drowsy', 2: 'LowVigilant'}  (alphabetical)
    # We need: YOLO_idx → our_idx
    yolo_to_ours = {}
    for yolo_idx, yolo_name in model.names.items():
        our_idx = CLASS_NAMES.index(yolo_name)
        yolo_to_ours[yolo_idx] = our_idx

    print(f'\n{"─"*60}')
    print(f'Fold {fold_idx}  |  test subjects = {FOLDS[fold_idx]}')
    print(f'  YOLO class mapping: {model.names}')
    print(f'  YOLO→Ours mapping : {yolo_to_ours}')
    print(f'  Our CLASS_NAMES   : {CLASS_NAMES}')

    # ── Evaluate on validation set ────────────────────────────────────────
    fold_dir = YOLO_DIR / f'fold_{fold_idx}'
    y_true, y_pred = [], []

    for cn_idx, cn in enumerate(CLASS_NAMES):
        class_dir = fold_dir / 'val' / cn
        if not class_dir.exists():
            continue
        img_list = sorted(class_dir.glob('*.jpg'))
        if not img_list:
            continue

        # Batch prediction on entire directory
        preds = model.predict(
            source=str(class_dir),
            verbose=False,
            batch=YOLO_BATCH,
        )
        for p in preds:
            yolo_pred_idx = int(p.probs.top1)     # YOLO's alphabetical index
            our_pred_idx  = yolo_to_ours[yolo_pred_idx]  # mapped to our index
            y_true.append(cn_idx)       # ground truth (our index)
            y_pred.append(our_pred_idx) # prediction (mapped to our index)

    # ── Metrics ───────────────────────────────────────────────────────────
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)

    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Macro-F1  : {f1:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  Samples   : {len(y_true)}')

    # Per-class report
    print(classification_report(
        y_true, y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    ))

    # Confusion matrix for this fold
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    print(f'  Confusion matrix:')
    print(f'               Pred_Alert  Pred_LowVig  Pred_Drowsy')
    for i, cn in enumerate(CLASS_NAMES):
        print(f'  True_{cn:11s}  {cm[i,0]:7d}  {cm[i,1]:11d}  {cm[i,2]:11d}')

    fold_result = {
        'fold': fold_idx,
        'accuracy': acc,
        'macro_f1': f1,
        'precision': prec,
        'recall': rec,
        'y_true': y_true,
        'y_pred': y_pred,
        'test_subjects': [s for s in FOLDS[fold_idx] if s not in NO_VIDEO],
        'n_train': len(meta_df[~meta_df['subject'].isin(FOLDS[fold_idx])]),
        'n_test': len(y_true),
    }
    all_fold_results.append(fold_result)

# ── Aggregate results ─────────────────────────────────────────────────────────
if all_fold_results:
    accs = [f['accuracy'] for f in all_fold_results]
    f1s  = [f['macro_f1'] for f in all_fold_results]
    precs = [f['precision'] for f in all_fold_results]
    recs  = [f['recall'] for f in all_fold_results]

    results = {
        'model_name': MODEL_NAME,
        'mean_acc': float(np.mean(accs)),
        'std_acc':  float(np.std(accs)),
        'mean_f1':  float(np.mean(f1s)),
        'std_f1':   float(np.std(f1s)),
        'mean_precision': float(np.mean(precs)),
        'mean_recall': float(np.mean(recs)),
        'folds': all_fold_results,
    }

    print(f'\n{"═"*60}')
    print(f'=== M5 YOLOv8-cls Aggregate Results ===')
    print(f'Accuracy  : {results["mean_acc"]:.4f} ± {results["std_acc"]:.4f}')
    print(f'Macro-F1  : {results["mean_f1"]:.4f} ± {results["std_f1"]:.4f}')
    print(f'Precision : {results["mean_precision"]:.4f}')
    print(f'Recall    : {results["mean_recall"]:.4f}')
    print(f'{"═"*60}')
else:
    print('\n⚠ No folds evaluated! Check that best.pt files exist.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: VISUALIZATIONS — Bar charts + Confusion matrices
# ══════════════════════════════════════════════════════════════════════════════

if not all_fold_results:
    print('No results to visualize.')
else:
    # ── Per-fold bar charts ───────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    folds_x = [f'Fold {f["fold"]}' for f in all_fold_results]
    accs    = [f['accuracy'] for f in all_fold_results]
    f1s     = [f['macro_f1'] for f in all_fold_results]

    axes[0].bar(folds_x, accs, color='steelblue', edgecolor='white')
    axes[0].axhline(np.mean(accs), color='red', ls='--', label=f'Mean={np.mean(accs):.3f}')
    for i, v in enumerate(accs):
        axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    axes[0].set_title('M5 — Per-fold Accuracy')
    axes[0].set_ylim(0, 1)
    axes[0].legend()

    axes[1].bar(folds_x, f1s, color='darkorange', edgecolor='white')
    axes[1].axhline(np.mean(f1s), color='red', ls='--', label=f'Mean={np.mean(f1s):.3f}')
    for i, v in enumerate(f1s):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    axes[1].set_title('M5 — Per-fold Macro-F1')
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(RESULT_DIR / 'M5_fold_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Aggregate confusion matrix ────────────────────────────────────────
    all_true = []
    all_pred = []
    for f in all_fold_results:
        all_true.extend(f['y_true'])
        all_pred.extend(f['y_pred'])

    cm = confusion_matrix(all_true, all_pred, labels=[0, 1, 2])

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    axes[0].set_title('M5 — Confusion Matrix (all folds)')

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    axes[1].set_title('M5 — Normalised Confusion Matrix')

    plt.tight_layout()
    plt.savefig(RESULT_DIR / 'M5_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10: RESULTS TABLE — Per-fold + aggregate summary
# ══════════════════════════════════════════════════════════════════════════════

if not all_fold_results:
    print('No results available.')
else:
    rows = []
    for f in all_fold_results:
        rows.append({
            'Fold': f['fold'],
            'Test Subjects': ', '.join(f['test_subjects']),
            'Train': f['n_train'],
            'Test': f['n_test'],
            'Accuracy': f'{f["accuracy"]:.4f}',
            'Macro-F1': f'{f["macro_f1"]:.4f}',
            'Precision': f'{f["precision"]:.4f}',
            'Recall': f'{f["recall"]:.4f}',
        })

    df_results = pd.DataFrame(rows)

    # Add mean/std rows
    accs = [f['accuracy'] for f in all_fold_results]
    f1s  = [f['macro_f1'] for f in all_fold_results]
    precs = [f['precision'] for f in all_fold_results]
    recs  = [f['recall'] for f in all_fold_results]

    mean_row = pd.DataFrame([{
        'Fold': 'Mean', 'Test Subjects': '',
        'Train': '', 'Test': '',
        'Accuracy': f'{np.mean(accs):.4f}',
        'Macro-F1': f'{np.mean(f1s):.4f}',
        'Precision': f'{np.mean(precs):.4f}',
        'Recall': f'{np.mean(recs):.4f}',
    }])
    std_row = pd.DataFrame([{
        'Fold': '±Std', 'Test Subjects': '',
        'Train': '', 'Test': '',
        'Accuracy': f'{np.std(accs):.4f}',
        'Macro-F1': f'{np.std(f1s):.4f}',
        'Precision': f'{np.std(precs):.4f}',
        'Recall': f'{np.std(recs):.4f}',
    }])

    df_results = pd.concat([df_results, mean_row, std_row], ignore_index=True)
    display(df_results)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11: SAVE RESULTS + WEIGHTS → Google Drive
# ══════════════════════════════════════════════════════════════════════════════
# Saves:
#   1. M5_results.json — full results for notebook 10 comparison
#   2. best.pt files   — trained weights for each fold
#   3. Plots           — PNG charts

def to_serializable(obj):
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, dict):           return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):           return [to_serializable(i) for i in obj]
    return obj

if not all_fold_results:
    print('No results to save.')
else:
    # ── Save JSON ─────────────────────────────────────────────────────────
    # Save locally
    json_path = RESULT_DIR / f'{MODEL_NAME}_results.json'
    with open(json_path, 'w') as f:
        json.dump(to_serializable(results), f, indent=2)
    print(f'✓ Results JSON → {json_path}')

    # Save to Drive
    drive_json = DRIVE_OUTPUT / f'{MODEL_NAME}_results.json'
    shutil.copy2(str(json_path), str(drive_json))
    print(f'✓ Results JSON → {drive_json}')

    # ── Copy best.pt weights to Drive ─────────────────────────────────────
    weights_dir = DRIVE_OUTPUT / 'weights'
    weights_dir.mkdir(parents=True, exist_ok=True)

    for fold_idx in range(len(FOLDS)):
        src = CKPT_DIR / f'M5_fold{fold_idx}' / 'weights' / 'best.pt'
        if src.exists():
            dst = weights_dir / f'M5_fold{fold_idx}_best.pt'
            shutil.copy2(str(src), str(dst))
            size_mb = dst.stat().st_size / 1e6
            print(f'✓ Fold {fold_idx} best.pt ({size_mb:.1f} MB) → {dst}')
        else:
            print(f'✗ Fold {fold_idx} best.pt not found')

    # ── Copy plots to Drive ───────────────────────────────────────────────
    for png in RESULT_DIR.glob('*.png'):
        shutil.copy2(str(png), str(DRIVE_OUTPUT / png.name))
        print(f'✓ {png.name} → Drive')

    print(f'\n{"═"*60}')
    print(f'All outputs saved to: {DRIVE_OUTPUT}')
    print(f'\nTo use on your local PC:')
    print(f'  1. Download weights from Drive: {weights_dir}')
    print(f'  2. Place M5_foldX_best.pt in local:')
    print(f'     models/checkpoints/yolo_cls/M5_foldX/weights/best.pt')
    print(f'  3. Download {MODEL_NAME}_results.json to results/reports/')
    print(f'{"═"*60}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12: VERIFY — Quick sanity check on saved weights
# ══════════════════════════════════════════════════════════════════════════════

print('Verifying saved weights on Drive ...')
weights_dir = DRIVE_OUTPUT / 'weights'

for fold_idx in range(len(FOLDS)):
    pt_path = weights_dir / f'M5_fold{fold_idx}_best.pt'
    if pt_path.exists():
        model = YOLO(str(pt_path))
        print(f'  Fold {fold_idx}: ✓ loaded  |  classes={model.names}')
    else:
        print(f'  Fold {fold_idx}: ✗ missing')

print(f'\n=== Final M5 Results ===')
print(f'Accuracy  : {results["mean_acc"]:.4f} ± {results["std_acc"]:.4f}')
print(f'Macro-F1  : {results["mean_f1"]:.4f} ± {results["std_f1"]:.4f}')
print(f'Precision : {results["mean_precision"]:.4f}')
print(f'Recall    : {results["mean_recall"]:.4f}')
print('\nDone! Download weights + results from Google Drive.')